# Step 20.2 — Save Optimizer State

本节只覆盖 Adam moments 与 step count，承接 20.1 的模型权重保存。20.1 实际提交 `317b0bdcc25866b5db012f7e15fd46ae7c689999` 已独立核心静态验收 PASS，初始修正 0/2；19.5 为 2/2，19.2/19.3 各 1/2。这些是材料验收，不代表学员掌握或运行反馈。

本 Notebook 所有代码单元均未执行、无输出。本轮不保存真实状态文件、不加载状态、不训练。20.2 初始生成 0/2；完成即停在 20.2，不进入 20.3。

## 1. 为什么只保存权重不够？

模型权重 `θ` 决定当前 forward；Adam 还依赖优化器状态。常见状态包含一阶矩 (m_t)、二阶矩 (v_t) 与更新步数 (t)。

[
m_t=eta_1m_{t-1}+(1-eta_1)g_t,qquad
v_t=eta_2v_{t-1}+(1-eta_2)g_t^2.
]

其中 (g_t) 是当前梯度，平方按元素计算。Adam 的更新还使用历史矩和步数进行偏差修正（若启用）。所以仅恢复 (	heta_t) 而将 Adam 状态重新置零，下一次更新通常不会与原训练轨迹相同。

本节保存的是**optimizer state**，不把它误称为完整训练 checkpoint。超参数、随机数状态、sampler位置、数据split、训练进度与模型结构配置仍需单独记录。

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json

import mlx.core as mx
import mlx.nn as nn
import mlx.optimizers as optim
from mlx.utils import tree_flatten, tree_unflatten

## 2. 完整模型结构与匹配的对象来源

本节沿用 Step 15.4 的完整 MiniGPT 类。模型与 optimizer 必须来自同一次 setup：先构造指定结构的模型，再创建与其 trainable parameters 对应的 Adam 状态。不能拿另一模型的 m/v 状态直接装入当前模型。

真实词表大小 (V) 必须由项目同一字符表/token-ID映射提供。本节不读取语料，因此显式要求工作副本使用者提供它，不用 toy 数值代替。

In [ ]:
MODEL_CONFIG = {
    "vocab_size": None,  # Must come from the real token-ID mapping.
    "dims": 64,
    "num_heads": 4,
    "num_blocks": 2,
    "max_seq_len": 64,
}

def make_model_kwargs(config):
    if not isinstance(config["vocab_size"], int) or config["vocab_size"] < 1:
        raise ValueError("Set vocab_size from the real token-ID mapping.")
    if config["dims"] % config["num_heads"] != 0:
        raise ValueError("dims must be divisible by num_heads.")
    return dict(config)

In [ ]:
import math
import mlx.core as mx
import mlx.nn as nn


class CausalMultiHeadAttention(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()

        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        self.dims = dims
        self.num_heads = num_heads
        self.head_dim = dims // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)  # Python float, not trainable

        self.qkv_proj = nn.Linear(
            input_dims=dims,
            output_dims=3 * dims,
            bias=False,
        )
        self.c_proj = nn.Linear(
            input_dims=dims,
            output_dims=dims,
            bias=False,
        )

    def __call__(self, x: mx.array) -> mx.array:
        if len(x.shape) != 3:
            raise ValueError('attention input must have shape (B,T,C)')

        B, T, C = x.shape
        if C != self.dims:
            raise ValueError('attention input width must equal dims')

        H = self.num_heads
        D = self.head_dim

        qkv = self.qkv_proj(x)  # (B,T,3C)
        q, k, v = mx.split(qkv, indices_or_sections=3, axis=-1)

        q = mx.reshape(q, (B, T, H, D))
        k = mx.reshape(k, (B, T, H, D))
        v = mx.reshape(v, (B, T, H, D))

        q = mx.swapaxes(q, 1, 2)  # (B,H,T,D)
        k = mx.swapaxes(k, 1, 2)
        v = mx.swapaxes(v, 1, 2)

        scores = mx.matmul(
            q,
            mx.swapaxes(k, -1, -2),
        ) * self.scale  # (B,H,T,T)

        # IMPORTANT: structural mask is local to forward.
        # It is NOT stored as self.causal_mask.
        future = mx.triu(mx.ones((T, T)), k=1) > 0
        mask = mx.where(
            future,
            mx.array(float('-inf'), dtype=scores.dtype),
            mx.array(0.0, dtype=scores.dtype),
        )

        weights = mx.softmax(scores + mask, axis=-1)
        head_out = mx.matmul(weights, v)  # (B,H,T,D)

        token_major = mx.swapaxes(head_out, 1, 2)  # (B,T,H,D)
        merged = mx.reshape(token_major, (B, T, C))
        return self.c_proj(merged)  # (B,T,C)


class FeedForwardMLP(nn.Module):
    def __init__(self, dims: int):
        super().__init__()
        self.fc1 = nn.Linear(dims, 4 * dims, bias=True)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * dims, dims, bias=True)

    def __call__(self, x: mx.array) -> mx.array:
        return self.fc2(self.act(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.RMSNorm(dims=dims)
        self.attn = CausalMultiHeadAttention(dims=dims, num_heads=num_heads)
        self.norm2 = nn.RMSNorm(dims=dims)
        self.mlp = FeedForwardMLP(dims=dims)

    def __call__(self, x: mx.array) -> mx.array:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


# Definitions only. This master notebook does not execute them.


In [ ]:
class MiniGPT(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        dims: int,
        num_heads: int,
        num_blocks: int,
        max_seq_len: int,
    ):
        super().__init__()

        if vocab_size <= 0:
            raise ValueError('vocab_size must be positive')
        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if num_blocks <= 0:
            raise ValueError('num_blocks must be positive')
        if max_seq_len <= 0:
            raise ValueError('max_seq_len must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        # Python metadata: these are not trainable mx.array parameters.
        self.vocab_size = vocab_size
        self.dims = dims
        self.max_seq_len = max_seq_len

        # Learned lookup tables.
        self.token_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            dims=dims,
        )
        self.position_embedding = nn.Embedding(
            num_embeddings=max_seq_len,
            dims=dims,
        )

        # A list of distinct Module instances; MLX recursively tracks them.
        self.blocks = [
            TransformerBlock(
                dims=dims,
                num_heads=num_heads,
            )
            for _ in range(num_blocks)
        ]

        self.final_norm = nn.RMSNorm(dims=dims)
        self.lm_head = nn.Linear(
            input_dims=dims,
            output_dims=vocab_size,
            bias=False,
        )

    def __call__(self, token_ids: mx.array) -> mx.array:
        if len(token_ids.shape) != 2:
            raise ValueError('token_ids must have shape (B,T)')

        B, T = token_ids.shape

        if T < 1:
            raise ValueError('sequence length T must be at least 1')
        if T > self.max_seq_len:
            raise ValueError(
                f'sequence length {T} exceeds max_seq_len {self.max_seq_len}'
            )

        # Forward-local structural indices: not Module parameters.
        position_ids = mx.arange(T)  # (T,)

        token_features = self.token_embedding(token_ids)        # (B,T,C)
        position_features = self.position_embedding(position_ids)  # (T,C)
        x = token_features + position_features                  # (B,T,C)

        for block in self.blocks:
            x = block(x)  # invariant: (B,T,C)

        x = self.final_norm(x)  # (B,T,C)
        logits = self.lm_head(x)  # (B,T,V)

        expected_shape = (B, T, self.vocab_size)
        if logits.shape != expected_shape:
            raise RuntimeError(
                f'logits shape {logits.shape} != expected {expected_shape}'
            )

        return logits


# Definition only. No MiniGPT forward is executed in this master notebook.


## 3. 构造 Adam 状态：零状态不是训练结果

本节显式使用课程设置 `bias_correction=True`；固定 MLX v0.32.2 的 Adam 默认值为 `False`。初始化后 Adam 的 moments 为对应参数 shape 的零数组，step 从初始状态开始。这是新的优化器状态，不是已训练历史。

若模型权重来自某个训练时刻，而 Adam 却刚初始化为零状态，那么这两者不匹配，不能称为精确续训。

In [ ]:
ADAM_CONFIG = {
    "learning_rate": 3e-4,
    "betas": [0.9, 0.999],
    "eps": 1e-8,
    "bias_correction": True,  # Explicit course choice; MLX v0.32.2 default is False.
}

def create_model_and_optimizer(vocab_size):
    config = dict(MODEL_CONFIG)
    config["vocab_size"] = vocab_size
    model = MiniGPT(**make_model_kwargs(config))
    optimizer = optim.Adam(**ADAM_CONFIG)
    optimizer.init(model.trainable_parameters())
    return model, optimizer

# Future work-copy use only; no model is created in this published master.
# model, optimizer = create_model_and_optimizer(real_vocab_size)

## 4. State tree、shape 与标量 step

MLX optimizer 的 `state` 是嵌套状态树。对 Adam 来说，moment arrays 的结构与模型参数树对应；每个参数张量都有相同 shape 的一阶矩和二阶矩。比如某个权重 shape 是 ((192,64))，其对应的 (m) 与 (v) 也各是 ((192,64))。`step` 则是更新计数，不是模型参数数组。

`tree_flatten` 把树展成带路径的叶子序列；`tree_unflatten` 使用相同路径重建结构。本节将数组叶子存进 NPZ，将 Python 标量元数据（例如 step）放进 JSON manifest。路径列表用来保证恢复时树结构和顺序一致。

In [ ]:
def flatten_optimizer_state(state):
    flattened = tree_flatten(state)
    array_items = []
    scalar_items = []
    for path, value in flattened:
        if isinstance(value, mx.array):
            array_items.append((str(path), value))
        elif isinstance(value, (int, float, bool, str)) or value is None:
            scalar_items.append({"path": str(path), "value": value})
        else:
            raise TypeError(f"Unsupported optimizer-state leaf at {path!r}: {type(value)!r}")
    return array_items, scalar_items

## 5. 保存优化器状态：数组与 manifest 分开

**Serialization（序列化）**把内存中的状态转换为可保存的表示。本节采用两个文件：

- `optimizer_state.npz`：所有数组叶子。
- `optimizer_state.json`：每个数组的键与树路径、标量状态（包括 step）、格式版本和必要说明。

保存前对状态树执行 `mx.eval(optimizer.state)`，确保数组依赖完成。输出目录必须是新目录，使用 `mkdir(exist_ok=False)` 防止覆盖旧状态。这里不保存真实文件；代码仅供未来工作副本使用。

In [ ]:
def save_optimizer_state_to_new_directory(optimizer, output_root):
    root = Path(output_root) / "optimizer_states"
    root.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    output_dir = root / f"adam_state_{stamp}"
    output_dir.mkdir(exist_ok=False)

    mx.eval(optimizer.state)
    array_items, scalar_items = flatten_optimizer_state(optimizer.state)
    arrays = {}
    array_manifest = []
    for index, (path, value) in enumerate(array_items):
        key = f"state_array_{index:04d}"
        arrays[key] = value
        array_manifest.append({"key": key, "path": path, "shape": list(value.shape), "dtype": str(value.dtype)})

    arrays_path = output_dir / "optimizer_state.npz"
    manifest_path = output_dir / "optimizer_state.json"
    mx.savez(str(arrays_path), **arrays)
    manifest = {
        "format_version": 1,
        "optimizer_class": type(optimizer).__name__,
        "array_file": arrays_path.name,
        "arrays": array_manifest,
        "scalars": scalar_items,
        "note": "Optimizer state only; config/model weights/data/RNG are separate.",
    }
    with manifest_path.open("x", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)
    return arrays_path, manifest_path

# Definition only. No optimizer state is saved during material preparation.

## 6. 恢复时必须匹配状态结构

加载的目标必须是与保存时一致的模型结构，并已创建对应的 optimizer。先读 manifest 和 NPZ 数组，再按记录路径重建状态树，最后赋回 optimizer 的 state。这里展示核心映射，不调用它，也不声称完成精确恢复。

对于不同 width、层数、词表或参数命名的模型，状态路径或 shape 不匹配时必须停止，而不是静默忽略。

In [ ]:
def restore_optimizer_state_from_files(optimizer, arrays_path, manifest_path):
    with Path(manifest_path).open("r", encoding="utf-8") as f:
        manifest = json.load(f)
    if manifest.get("format_version") != 1:
        raise ValueError("Unsupported optimizer-state format version.")
    if manifest.get("optimizer_class") != type(optimizer).__name__:
        raise ValueError("Optimizer class does not match saved state.")

    archive = mx.load(str(arrays_path))
    path_to_value = {}
    for item in manifest["arrays"]:
        value = archive[item["key"]]
        if list(value.shape) != item["shape"] or str(value.dtype) != item["dtype"]:
            raise ValueError(f"Array metadata mismatch at {item['path']}")
        path_to_value[item["path"]] = value
    for item in manifest["scalars"]:
        path_to_value[item["path"]] = item["value"]

    # Rebuild from the current optimizer's tree paths to reject missing/extra paths.
    current_flat = tree_flatten(optimizer.state)
    current_paths = [str(path) for path, _ in current_flat]
    if set(current_paths) != set(path_to_value):
        raise ValueError("Saved optimizer-state tree paths do not match the current optimizer.")
    rebuilt = tree_unflatten([(path, path_to_value[str(path)]) for path, _ in current_flat])
    optimizer.state = rebuilt
    mx.eval(optimizer.state)

# Definition only. No state is loaded during material preparation.

## 7. 保存内容的边界

| 信息 | 本节优化器状态文件 |
|---|---|
| Adam 一阶矩 (m) | 保存为数组叶子 |
| Adam 二阶矩 (v) | 保存为数组叶子 |
| Adam step | 保存为标量元数据（若在状态树中） |
| 模型权重 (	heta) | 不包含 |
| Adam 超参数 | 只记录优化器类；配置仍需另存 |
| 模型结构与词表映射 | 不包含 |
| Python/MLX RNG 与 sampler 状态 | 不包含 |
| 数据 split、训练历史与基线 JSON | 不包含 |

要恢复同一优化过程，至少需要匹配的模型权重和 optimizer state；还要确保模型结构、Adam 超参数、数据、随机状态与训练进度等信息一致。20.2只聚焦 optimizer state 的序列化，不宣称完整训练恢复，完整 checkpoint 设计留待后续小节。

## 本节边界与计数

本节初始生成 0/2。20.1 保持 0/2；19.5 为 2/2；19.2/19.3 各为 1/2；其余历史计数不重置。

所有代码单元未执行、无输出。本轮没有创建模型、初始化/保存/加载真实 optimizer state、训练、验证、读取/下载语料或生成 checkpoint。完成即停在 20.2，不进入 20.3；后续按请求推进至大 Step 20 全部材料完成后停止，不进入 21。